In [0]:
import re

storage_account = "group3gpsa"

source_dir = f"abfss://landing@{storage_account}.dfs.core.windows.net/MonthlyNycData"
target_dir = f"abfss://landing@{storage_account}.dfs.core.windows.net/LoopedNYCData"

type_labels = {
    "yellow": "Yellow",
    "green": "Green",
    "fhv": "For Hire Vehicles",
    "fhvhv": "High Volume FHV"
}

pattern = re.compile(r"^(yellow|green|fhv|fhvhv)_tripdata_(2025|2026)\.parquet$", re.IGNORECASE)

all_items = dbutils.fs.ls(source_dir)
print(all_items)
source_files = []
for f in all_items:
    if f.isDir():
        continue
    m = pattern.match(f.name)
    if m:
        source_files.append((f.path, m.group(1).lower(), m.group(2)))

print(f"Found {len(source_files)} matching parquet files")

for src, trip_type, year in source_files:
    type_label = type_labels[trip_type]
    final_name = f"{year} {type_label} Taxi Trip Data"
    final_json = f"{target_dir}/{final_name}"
    temp_dir = f"{target_dir}/_tmp_{trip_type}_{year}"

    print(f"Starting: {src}, {trip_type}, {year}")
    print(f"Target:   {final_json}")

    df = spark.read.parquet(src)
    json_df = df.toJSON().toDF("value")

    try:
        dbutils.fs.rm(temp_dir, True)
    except:
        pass

    try:
        dbutils.fs.rm(final_json, True)
    except:
        pass

    (
        json_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .text(temp_dir)
    )

    written_files = dbutils.fs.ls(temp_dir)
    part_file = [f.path for f in written_files if f.name.startswith("part-")][0]

    dbutils.fs.mv(part_file, final_json)
    dbutils.fs.rm(temp_dir, True)

    print(f"Finished: {final_json}")

print("All done.")